In [2]:
import os
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

# ==== PATHS ====
BASE_DIR = "./"   # change if needed

PATHS = {
    "ml_movies": os.path.join(BASE_DIR, "ML25M", "movies.csv"),
    "ml_links": os.path.join(BASE_DIR, "ML25M", "links.csv"),
    "ml_ratings": os.path.join(BASE_DIR, "ML25M", "ratings.csv"),
    "ml_tags": os.path.join(BASE_DIR, "ML25M", "tags.csv"),
    "ml_genome_scores": os.path.join(BASE_DIR, "ML25M", "genome-scores.csv"),
    "ml_genome_tags": os.path.join(BASE_DIR, "ML25M", "genome-tags.csv"),

    "poster_csv": os.path.join(BASE_DIR, "MovieGenreFromItsPoster", "MovieGenre.csv"),

    "movies_metadata": os.path.join(BASE_DIR, "TheMoviesDataset", "movies_metadata.csv"),
    "credits": os.path.join(BASE_DIR, "TheMoviesDataset", "credits.csv"),
    "keywords": os.path.join(BASE_DIR, "TheMoviesDataset", "keywords.csv"),
    "tmdb_links": os.path.join(BASE_DIR, "TheMoviesDataset", "links.csv"),

    "tmdb_all_movies": os.path.join(BASE_DIR, "TMDB", "TMDB_all_movies.csv"),
}

In [3]:
# Some files may have mixed types, so low_memory=False helps
dfs = {}

for name, path in PATHS.items():
    try:
        dfs[name] = pd.read_csv(path, low_memory=False)
        print(f"[OK] {name}: {dfs[name].shape}")
    except Exception as e:
        print(f"[FAILED] {name}: {e}")

[OK] ml_movies: (62423, 3)
[OK] ml_links: (62423, 3)
[OK] ml_ratings: (25000095, 4)
[OK] ml_tags: (1093360, 4)
[OK] ml_genome_scores: (15584448, 3)
[OK] ml_genome_tags: (1128, 2)
[FAILED] poster_csv: 'utf-8' codec can't decode byte 0xed in position 17156: invalid continuation byte
[OK] movies_metadata: (45466, 24)
[OK] credits: (45476, 3)
[OK] keywords: (46419, 2)
[OK] tmdb_links: (45843, 3)
[OK] tmdb_all_movies: (1178829, 28)


In [4]:
poster_path = PATHS["poster_csv"]

encodings_to_try = ["latin1", "ISO-8859-1", "cp1252"]

poster_df = None
for enc in encodings_to_try:
    try:
        poster_df = pd.read_csv(poster_path, encoding=enc, low_memory=False)
        print(f"[OK] poster_csv loaded with encoding={enc}, shape={poster_df.shape}")
        break
    except Exception as e:
        print(f"[FAILED] encoding={enc}: {e}")

if poster_df is None:
    raise ValueError("Could not load poster_csv with tried encodings.")

dfs["poster_csv"] = poster_df

[OK] poster_csv loaded with encoding=latin1, shape=(40108, 6)


In [5]:
important = ["ml_movies", "ml_links", "ml_ratings", "poster_csv", "movies_metadata", "tmdb_links", "tmdb_all_movies"]

for name in important:
    if name not in dfs:
        continue
        
    df = dfs[name]
    print("=" * 120)
    print(f"{name}  | shape={df.shape}")
    print("Columns:")
    print(df.columns.tolist())
    print("\nHead:")
    display(df.head(3))
    print("\nMissing values:")
    display(df.isna().sum()[df.isna().sum() > 0].sort_values(ascending=False).head(20))
    print("\nDuplicate rows:", df.duplicated().sum())
    print()

ml_movies  | shape=(62423, 3)
Columns:
['movieId', 'title', 'genres']

Head:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance



Missing values:


Series([], dtype: int64)


Duplicate rows: 0

ml_links  | shape=(62423, 3)
Columns:
['movieId', 'imdbId', 'tmdbId']

Head:


...


Duplicate rows: 0

tmdb_all_movies  | shape=(1178829, 28)
Columns:
['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date', 'revenue', 'runtime', 'budget', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'tagline', 'genres', 'production_companies', 'production_countries', 'spoken_languages', 'cast', 'director', 'director_of_photography', 'writers', 'producers', 'music_composer', 'imdb_rating', 'imdb_votes', 'poster_path']

Head:


,id,title,vote_average,vote_count,status,release_date,revenue,runtime,budget,imdb_id,original_language,original_title,overview,popularity,tagline,genres,production_companies,production_countries,spoken_languages,cast,director,director_of_photography,writers,producers,music_composer,imdb_rating,imdb_votes,poster_path
0,2,Ariel,7.1,367.0,Released,1988-10-21,0.0,73.0,0.0,tt0094675,fi,Ariel,A Finnish man goes to the city to find a job after the mine where he worked is closed and his father commits suicide.,4.1418,NaN,"Comedy, Drama, Romance, Crime",Villealfa Filmproductions,Finland,suomi,"Tarja Keinänen, Marja Packalén, Kari Helaseppä, Matti Pellonpää, Jorma Markkula, Tomi Salmela, Pekka Wilen, Jaakko Talaskivi, Reijo Marin, Heikki Anttila, Eetu Hilkamo, Esko Salminen, Timo Harakka...",Aki Kaurismäki,Timo Salminen,Aki Kaurismäki,Aki Kaurismäki,NaN,7.4,9712.0,/ojDg0PGvs6R9xYFodRct2kdI6wC.jpg
1,3,Shadows in Paradise,7.3,435.0,Released,1986-10-17,0.0,74.0,0.0,tt0092149,fi,Varjoja paratiisissa,"Nikander, a rubbish collector and would-be entrepreneur, finds his plans for success dashed when his business associate dies. One evening, he meets Ilona, a down-on-her-luck cashier, in a local su...",2.7136,NaN,"Comedy, Drama, Romance",Villealfa Filmproductions,Finland,"svenska, suomi, English","Ari Korhonen, Mari Rantasila, Erkki Rissanen, Jukka Mäkinen, Haije Alanoja, Matti Pellonpää, Sirkka Silin, Pekka Laiho, Antti Ortamo, Eskil Mansikka, Aki Kaurismäki, Pentti Koski, Ulla Kuosmanen, ...",Aki Kaurismäki,Timo Salminen,Aki Kaurismäki,Mika Kaurismäki,NaN,7.4,8587.0,/nj01hspawPof0mJmlgfjuLyJuRN.jpg
2,5,Four Rooms,5.9,2816.0,Released,1995-12-09,4257354.0,98.0,4000000.0,tt0113101,en,Four Rooms,It's Ted the Bellhop's first night on the job...and the hotel's very unusual guests are about to place him in some outrageous predicaments. It seems that this evening's room service is serving up ...,3.1942,"Twelve outrageous guests. Four scandalous requests. And one lone bellhop, in his first day on the job, who's in for the wildest New year's Eve of his life.",Comedy,"Miramax, A Band Apart",United States of America,English,"Danny Verduzco, Quentin Tarantino, Marisa Tomei, Tim Roth, Alicia Witt, Amanda de Cadenet, Patricia Vonne, Ione Skye, Laura Rush, Bruce Willis, Kathy Griffin, Tamlyn Tomita, Jennifer Beals, Lili T...","Robert Rodriguez, Alexandre Rockwell, Quentin Tarantino, Allison Anders","Guillermo Navarro, Rodrigo García, Phil Parmet, Andrzej Sekula","Quentin Tarantino, Alexandre Rockwell, Robert Rodriguez, Allison Anders","Lawrence Bender, Quentin Tarantino, Alexandre Rockwell",Combustible Edison,6.7,116736.0,/75aHn1NOYXh4M7L5shoeQ6NGykP.jpg



Missing values:


music_composer             1045483
tagline                    1000422
director_of_photography     862376
producers                   776753
imdb_rating                 713547
imdb_votes                  713547
production_companies        611227
writers                     582353
imdb_id                     517907
production_countries        437214
spoken_languages            429587
cast                        374119
genres                      317328
poster_path                 288910
director                    190229
overview                    180069
release_date                123245
title                           10
original_title                   9
dtype: int64


Duplicate rows: 0



In [6]:
for name in ["ml_links", "poster_csv", "movies_metadata", "tmdb_links", "tmdb_all_movies"]:
    if name not in dfs:
        continue
        
    df = dfs[name]
    print("=" * 120)
    print(name)

    info_df = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "non_null": df.notna().sum().values,
        "nulls": df.isna().sum().values,
        "n_unique": [df[col].nunique(dropna=True) for col in df.columns]
    }).sort_values("nulls", ascending=False)

    display(info_df)

ml_links


,column,dtype,non_null,nulls,n_unique
2,tmdbId,float64,62316,107,62281
0,movieId,int64,62423,0,62423
1,imdbId,int64,62423,0,62423


poster_csv


,column,dtype,non_null,nulls,n_unique
5,Poster,object,39383,725,38654
4,Genre,object,39963,145,1307
3,IMDB Score,float64,40060,48,84
0,imdbId,int64,40108,0,39515
1,Imdb Link,object,40108,0,39515
2,Title,object,40108,0,39470


movies_metadata


,column,dtype,non_null,nulls,n_unique
1,belongs_to_collection,object,4494,40972,1698
4,homepage,object,7782,37684,7673
19,tagline,object,20412,25054,20283
9,overview,object,44512,954,44307
11,poster_path,object,45080,386,45024
16,runtime,float64,45203,263,353
18,status,object,45379,87,6
14,release_date,object,45379,87,17336
6,imdb_id,object,45449,17,45417
7,original_language,object,45455,11,92


tmdb_links


,column,dtype,non_null,nulls,n_unique
2,tmdbId,float64,45624,219,45594
0,movieId,int64,45843,0,45843
1,imdbId,int64,45843,0,45843


tmdb_all_movies


,column,dtype,non_null,nulls,n_unique
24,music_composer,object,133346,1045483,44912
14,tagline,object,178407,1000422,172958
21,director_of_photography,object,316453,862376,115298
23,producers,object,402076,776753,290438
26,imdb_votes,float64,465282,713547,20155
25,imdb_rating,float64,465282,713547,91
16,production_companies,object,567602,611227,266236
22,writers,object,596476,582353,416226
9,imdb_id,object,660922,517907,660922
17,production_countries,object,741615,437214,13024


In [7]:
keywords = ["imdb", "tmdb", "movie", "title", "poster", "genre", "id"]

for name in ["poster_csv", "movies_metadata", "tmdb_links", "tmdb_all_movies", "ml_links", "ml_movies"]:
    if name not in dfs:
        continue
        
    print("=" * 100)
    print(name)
    cols = dfs[name].columns.tolist()
    matched = [c for c in cols if any(k in c.lower() for k in keywords)]
    print(matched)

poster_csv
['imdbId', 'Imdb Link', 'Title', 'IMDB Score', 'Genre', 'Poster']
movies_metadata
['genres', 'id', 'imdb_id', 'original_title', 'poster_path', 'title', 'video']
tmdb_links
['movieId', 'imdbId', 'tmdbId']
tmdb_all_movies
['id', 'title', 'imdb_id', 'original_title', 'genres', 'imdb_rating', 'imdb_votes', 'poster_path']
ml_links
['movieId', 'imdbId', 'tmdbId']
ml_movies
['movieId', 'title', 'genres']


### Merging

In [8]:
import numpy as np
import pandas as pd

def normalize_imdb_numeric(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    x = x.replace("tt", "")
    x = x.replace(".0", "")
    if x == "" or x.lower() == "nan":
        return np.nan
    return x

def normalize_tmdb(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace(".0", "")
    if x == "" or x.lower() == "nan":
        return np.nan
    return x

# -------- ML movies --------
ml_movies_small = dfs["ml_movies"].copy()

# -------- ML links --------
ml_links_small = dfs["ml_links"].copy()
ml_links_small["imdb_id_clean"] = ml_links_small["imdbId"].apply(normalize_imdb_numeric)
ml_links_small["tmdb_id_clean"] = ml_links_small["tmdbId"].apply(normalize_tmdb)

# -------- Ratings --------
ml_ratings_small = dfs["ml_ratings"].copy()

# Optional: smaller sample for fast experimentation
# ml_ratings_small = ml_ratings_small.sample(500000, random_state=42)

# -------- Poster dataset --------
poster_small = dfs["poster_csv"].copy()
poster_small["imdb_id_clean"] = poster_small["imdbId"].apply(normalize_imdb_numeric)

poster_small = poster_small.rename(columns={
    "Title": "poster_title",
    "Genre": "poster_genres",
    "Poster": "poster_url",
    "IMDB Score": "poster_imdb_score"
})

poster_small = poster_small[[
    "imdb_id_clean", "poster_title", "poster_imdb_score", "poster_genres", "poster_url"
]]

# remove obvious duplicates by imdb id, keep first non-null row
poster_small = poster_small.sort_values(by=["imdb_id_clean"]).drop_duplicates(subset=["imdb_id_clean"], keep="first")

# -------- Movies metadata --------
meta_small = dfs["movies_metadata"].copy()

meta_small["imdb_id_clean"] = meta_small["imdb_id"].astype(str).str.replace("tt", "", regex=False)
meta_small["tmdb_id_clean"] = meta_small["id"].apply(normalize_tmdb)

meta_small = meta_small.rename(columns={
    "title": "meta_title",
    "genres": "meta_genres",
    "poster_path": "meta_poster_path",
    "vote_average": "meta_vote_average",
    "vote_count": "meta_vote_count",
    "popularity": "meta_popularity",
    "runtime": "meta_runtime",
    "release_date": "meta_release_date",
    "overview": "meta_overview",
    "original_language": "meta_original_language"
})

meta_small = meta_small[[
    "tmdb_id_clean", "imdb_id_clean", "meta_title", "original_title",
    "meta_genres", "meta_poster_path", "meta_vote_average", "meta_vote_count",
    "meta_popularity", "meta_runtime", "meta_release_date",
    "meta_overview", "meta_original_language"
]]

# some duplicates exist in movies_metadata
meta_small = meta_small.drop_duplicates(subset=["tmdb_id_clean", "imdb_id_clean"], keep="first")

print("ml_movies_small:", ml_movies_small.shape)
print("ml_links_small:", ml_links_small.shape)
print("ml_ratings_small:", ml_ratings_small.shape)
print("poster_small:", poster_small.shape)
print("meta_small:", meta_small.shape)

ml_movies_small: (62423, 3)
ml_links_small: (62423, 5)
ml_ratings_small: (25000095, 4)
poster_small: (39515, 5)
meta_small: (45436, 13)


In [9]:
ratings_base = (
    ml_ratings_small
    .merge(ml_movies_small, on="movieId", how="left")
    .merge(
        ml_links_small[["movieId", "imdb_id_clean", "tmdb_id_clean"]],
        on="movieId",
        how="left"
    )
)

print("ratings_base shape:", ratings_base.shape)
display(ratings_base.head())
display(ratings_base[["movieId", "title", "genres", "imdb_id_clean", "tmdb_id_clean"]].isna().sum())

ratings_base shape: (25000095, 8)


,userId,movieId,rating,timestamp,title,genres,imdb_id_clean,tmdb_id_clean
0,1,296,5.0,1147880044,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,110912,680
1,1,306,3.5,1147868817,Three Colors: Red (Trois couleurs: Rouge) (1994),Drama,111495,110
2,1,307,5.0,1147868828,Three Colors: Blue (Trois couleurs: Bleu) (1993),Drama,108394,108
3,1,665,5.0,1147878820,Underground (1995),Comedy|Drama|War,114787,11902
4,1,899,3.5,1147868510,Singin' in the Rain (1952),Comedy|Musical|Romance,45152,872


movieId             0
title               0
genres              0
imdb_id_clean       0
tmdb_id_clean    2431
dtype: int64

In [10]:
merged_df = (
    ratings_base
    .merge(poster_small, on="imdb_id_clean", how="left")
    .merge(meta_small, on="tmdb_id_clean", how="left", suffixes=("", "_meta"))
)

print("merged_df shape:", merged_df.shape)
display(merged_df.head())

missing_report = pd.Series({
    "missing_imdb_id_clean": merged_df["imdb_id_clean"].isna().sum(),
    "missing_tmdb_id_clean": merged_df["tmdb_id_clean"].isna().sum(),
    "missing_poster_url": merged_df["poster_url"].isna().sum(),
    "missing_meta_title": merged_df["meta_title"].isna().sum(),
    "missing_meta_overview": merged_df["meta_overview"].isna().sum(),
    "missing_meta_genres": merged_df["meta_genres"].isna().sum(),
})
display(missing_report.to_frame("count"))

merged_df shape: (25000095, 24)


,userId,movieId,rating,timestamp,title,genres,imdb_id_clean,tmdb_id_clean,poster_title,poster_imdb_score,poster_genres,poster_url,imdb_id_clean_meta,meta_title,original_title,meta_genres,meta_poster_path,meta_vote_average,meta_vote_count,meta_popularity,meta_runtime,meta_release_date,meta_overview,meta_original_language
0,1,296,5.0,1147880044,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,110912,680,Pulp Fiction (1994),8.9,Crime|Drama,"https://images-na.ssl-images-amazon.com/images/M/MV5BMTkxMTA5OTAzMl5BMl5BanBnXkFtZTgwNjA5MDc3NjE@._V1_UX182_CR0,0,182,268_AL_.jpg",0110912,Pulp Fiction,Pulp Fiction,"[{'id': 53, 'name': 'Thriller'}, {'id': 80, 'name': 'Crime'}]",/dM2w364MScsjFf8pfMbaWUcWrR.jpg,8.3,8670.0,140.950236,154.0,1994-09-10,"A burger-loving hit man, his philosophical partner, a drug-addled gangster's moll and a washed-up boxer converge in this sprawling, comedic crime caper. Their adventures unfurl in three stories th...",en
1,1,306,3.5,1147868817,Three Colors: Red (Trois couleurs: Rouge) (1994),Drama,111495,110,Three Colors: Red (1994),8.1,Drama|Mystery|Romance,"https://images-na.ssl-images-amazon.com/images/M/MV5BYTg1MmNiMjItMmY4Yy00ZDQ3LThjMzYtZGQ0ZTQzNTdkMGQ1L2ltYWdlL2ltYWdlXkEyXkFqcGdeQXVyMTQxNzMzNDI@._V1_UX182_CR0,0,182,268_AL_.jpg",0111495,Three Colors: Red,Trois couleurs : Rouge,"[{'id': 18, 'name': 'Drama'}, {'id': 9648, 'name': 'Mystery'}, {'id': 10749, 'name': 'Romance'}]",/77CFEssoKesi4zvtADEpIrSKhA3.jpg,7.8,246.0,7.832755,99.0,1994-05-27,Red This is the third film from the trilogy by Kieślowski. “Red” meaning brotherliness. Here Kieślowski masterly tells strange coincidentally linked stories in the most packed work.,fr
2,1,307,5.0,1147868828,Three Colors: Blue (Trois couleurs: Bleu) (1993),Drama,108394,108,Three Colors: Blue (1993),8.0,Drama|Music|Mystery,"https://images-na.ssl-images-amazon.com/images/M/MV5BZjYxMzU5ZGItOTk0Yy00OGU4LTkyMGQtYmQ3MjQ0ODRjOTJlL2ltYWdlL2ltYWdlXkEyXkFqcGdeQXVyMTQxNzMzNDI@._V1_UX182_CR0,0,182,268_AL_.jpg",0108394,Three Colors: Blue,Trois couleurs : Bleu,"[{'id': 18, 'name': 'Drama'}, {'id': 10402, 'name': 'Music'}, {'id': 9648, 'name': 'Mystery'}]",/ztIqf7yGmb4JRFiv2i62K6PhTQR.jpg,7.7,311.0,8.843517,98.0,1993-01-10,A woman struggles to find a way to live her life after the death of her husband and child.,fr
3,1,665,5.0,1147878820,Underground (1995),Comedy|Drama|War,114787,11902,Underground (1995),8.1,Comedy|Drama|War,"https://images-na.ssl-images-amazon.com/images/M/MV5BMTgxMjI0MzY4MV5BMl5BanBnXkFtZTcwMzYxMzQyMQ@@._V1_UY268_CR3,0,182,268_AL_.jpg",0114787,Underground,Podzemlje,"[{'id': 10752, 'name': 'War'}, {'id': 18, 'name': 'Drama'}, {'id': 35, 'name': 'Comedy'}]",/bnrLsunhGI5FO316sSThtAHikdV.jpg,7.5,143.0,14.587894,170.0,1995-04-11,"Black marketeers Marko (Miki Manojlovic) and Blacky (Lazar Ristovski) manufacture and sell weapons to the Communist resistance in WWII Belgrade, living the good life along the way. Marko's surreal...",sr
4,1,899,3.5,1147868510,Singin' in the Rain (1952),Comedy|Musical|Romance,45152,872,Singin' in the Rain (1952),8.3,Comedy|Musical|Romance,"https://images-na.ssl-images-amazon.com/images/M/MV5BZDRjNGViMjQtOThlMi00MTA3LThkYzQtNzJkYjBkMGE0YzE1XkEyXkFqcGdeQXVyNDYyMDk5MTU@._V1_UY268_CR0,0,182,268_AL_.jpg",0045152,Singin' in the Rain,Singin' in the Rain,"[{'id': 35, 'name': 'Comedy'}, {'id': 10402, 'name': 'Music'}, {'id': 10749, 'name': 'Romance'}]",/d5J53CwrVs6txB8zhE6qS2QhIV.jpg,7.9,747.0,11.064858,103.0,1952-04-10,"In 1927 Hollywood, Don Lockwood and Lina Lamont are a famous on-screen romantic pair in silent movies, but Lina mistakes the on-screen romance for real love. When their latest film is transformed ...",en


,count
missing_imdb_id_clean,0
missing_tmdb_id_clean,2431
missing_poster_url,449433
missing_meta_title,267543
missing_meta_overview,272179
missing_meta_genres,267515


In [11]:
coverage = pd.DataFrame({
    "feature": [
        "poster match",
        "metadata match",
        "overview available",
        "metadata genres available"
    ],
    "matched_rows": [
        merged_df["poster_url"].notna().sum(),
        merged_df["meta_title"].notna().sum(),
        merged_df["meta_overview"].notna().sum(),
        merged_df["meta_genres"].notna().sum()
    ]
})

coverage["total_rows"] = len(merged_df)
coverage["coverage_percent"] = (coverage["matched_rows"] / coverage["total_rows"] * 100).round(2)
display(coverage)

,feature,matched_rows,total_rows,coverage_percent
0,poster match,24550662,25000095,98.20
1,metadata match,24732552,25000095,98.93
2,overview available,24727916,25000095,98.91
3,metadata genres available,24732580,25000095,98.93


In [12]:
movie_feature_df = (
    merged_df[[
        "movieId", "title", "genres", "imdb_id_clean", "tmdb_id_clean",
        "poster_title", "poster_imdb_score", "poster_genres", "poster_url",
        "meta_title", "original_title", "meta_genres", "meta_poster_path",
        "meta_vote_average", "meta_vote_count", "meta_popularity",
        "meta_runtime", "meta_release_date", "meta_overview", "meta_original_language"
    ]]
    .drop_duplicates(subset=["movieId"])
    .reset_index(drop=True)
)

print("movie_feature_df shape:", movie_feature_df.shape)
display(movie_feature_df.head())

movie_feature_df shape: (59047, 20)


,movieId,title,genres,imdb_id_clean,tmdb_id_clean,poster_title,poster_imdb_score,poster_genres,poster_url,meta_title,original_title,meta_genres,meta_poster_path,meta_vote_average,meta_vote_count,meta_popularity,meta_runtime,meta_release_date,meta_overview,meta_original_language
0,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,110912,680,Pulp Fiction (1994),8.9,Crime|Drama,"https://images-na.ssl-images-amazon.com/images/M/MV5BMTkxMTA5OTAzMl5BMl5BanBnXkFtZTgwNjA5MDc3NjE@._V1_UX182_CR0,0,182,268_AL_.jpg",Pulp Fiction,Pulp Fiction,"[{'id': 53, 'name': 'Thriller'}, {'id': 80, 'name': 'Crime'}]",/dM2w364MScsjFf8pfMbaWUcWrR.jpg,8.3,8670.0,140.950236,154.0,1994-09-10,"A burger-loving hit man, his philosophical partner, a drug-addled gangster's moll and a washed-up boxer converge in this sprawling, comedic crime caper. Their adventures unfurl in three stories th...",en
1,306,Three Colors: Red (Trois couleurs: Rouge) (1994),Drama,111495,110,Three Colors: Red (1994),8.1,Drama|Mystery|Romance,"https://images-na.ssl-images-amazon.com/images/M/MV5BYTg1MmNiMjItMmY4Yy00ZDQ3LThjMzYtZGQ0ZTQzNTdkMGQ1L2ltYWdlL2ltYWdlXkEyXkFqcGdeQXVyMTQxNzMzNDI@._V1_UX182_CR0,0,182,268_AL_.jpg",Three Colors: Red,Trois couleurs : Rouge,"[{'id': 18, 'name': 'Drama'}, {'id': 9648, 'name': 'Mystery'}, {'id': 10749, 'name': 'Romance'}]",/77CFEssoKesi4zvtADEpIrSKhA3.jpg,7.8,246.0,7.832755,99.0,1994-05-27,Red This is the third film from the trilogy by Kieślowski. “Red” meaning brotherliness. Here Kieślowski masterly tells strange coincidentally linked stories in the most packed work.,fr
2,307,Three Colors: Blue (Trois couleurs: Bleu) (1993),Drama,108394,108,Three Colors: Blue (1993),8.0,Drama|Music|Mystery,"https://images-na.ssl-images-amazon.com/images/M/MV5BZjYxMzU5ZGItOTk0Yy00OGU4LTkyMGQtYmQ3MjQ0ODRjOTJlL2ltYWdlL2ltYWdlXkEyXkFqcGdeQXVyMTQxNzMzNDI@._V1_UX182_CR0,0,182,268_AL_.jpg",Three Colors: Blue,Trois couleurs : Bleu,"[{'id': 18, 'name': 'Drama'}, {'id': 10402, 'name': 'Music'}, {'id': 9648, 'name': 'Mystery'}]",/ztIqf7yGmb4JRFiv2i62K6PhTQR.jpg,7.7,311.0,8.843517,98.0,1993-01-10,A woman struggles to find a way to live her life after the death of her husband and child.,fr
3,665,Underground (1995),Comedy|Drama|War,114787,11902,Underground (1995),8.1,Comedy|Drama|War,"https://images-na.ssl-images-amazon.com/images/M/MV5BMTgxMjI0MzY4MV5BMl5BanBnXkFtZTcwMzYxMzQyMQ@@._V1_UY268_CR3,0,182,268_AL_.jpg",Underground,Podzemlje,"[{'id': 10752, 'name': 'War'}, {'id': 18, 'name': 'Drama'}, {'id': 35, 'name': 'Comedy'}]",/bnrLsunhGI5FO316sSThtAHikdV.jpg,7.5,143.0,14.587894,170.0,1995-04-11,"Black marketeers Marko (Miki Manojlovic) and Blacky (Lazar Ristovski) manufacture and sell weapons to the Communist resistance in WWII Belgrade, living the good life along the way. Marko's surreal...",sr
4,899,Singin' in the Rain (1952),Comedy|Musical|Romance,45152,872,Singin' in the Rain (1952),8.3,Comedy|Musical|Romance,"https://images-na.ssl-images-amazon.com/images/M/MV5BZDRjNGViMjQtOThlMi00MTA3LThkYzQtNzJkYjBkMGE0YzE1XkEyXkFqcGdeQXVyNDYyMDk5MTU@._V1_UY268_CR0,0,182,268_AL_.jpg",Singin' in the Rain,Singin' in the Rain,"[{'id': 35, 'name': 'Comedy'}, {'id': 10402, 'name': 'Music'}, {'id': 10749, 'name': 'Romance'}]",/d5J53CwrVs6txB8zhE6qS2QhIV.jpg,7.9,747.0,11.064858,103.0,1952-04-10,"In 1927 Hollywood, Don Lockwood and Lina Lamont are a famous on-screen romantic pair in silent movies, but Lina mistakes the on-screen romance for real love. When their latest film is transformed ...",en


In [13]:
print("Unique users:", merged_df["userId"].nunique())
print("Unique movies in ratings:", merged_df["movieId"].nunique())
print("Movies with poster:", movie_feature_df["poster_url"].notna().sum())
print("Movies with metadata overview:", movie_feature_df["meta_overview"].notna().sum())
print("Movies with both poster + metadata:", ((movie_feature_df["poster_url"].notna()) & (movie_feature_df["meta_overview"].notna())).sum())

display(
    movie_feature_df[[
        "movieId", "title", "poster_url", "meta_title", "meta_overview", "meta_genres"
    ]].sample(10, random_state=42)
)

Unique users: 162541
Unique movies in ratings: 59047
Movies with poster: 35912
Movies with metadata overview: 41477
Movies with both poster + metadata: 35146


,movieId,title,poster_url,meta_title,meta_overview,meta_genres
24367,117178,Amira & Sam (2014),"https://images-na.ssl-images-amazon.com/images/M/MV5BMjM5MTE1MDY0Nl5BMl5BanBnXkFtZTgwOTE3NTY4MzE@._V1_UX182_CR0,0,182,268_AL_.jpg",Amira & Sam,"Sam, a soldier who had served in Afghanistan and Iraq, meets Amira when he visits her uncle, Bassam, who had served as Sam's Iraqi translator. Bassam and Sam have a special bond due to their time ...","[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'name': 'Romance'}, {'id': 35, 'name': 'Comedy'}]"
10121,4759,Two Can Play That Game (2001),"https://images-na.ssl-images-amazon.com/images/M/MV5BMjAwOTU5MTk2M15BMl5BanBnXkFtZTcwMDY5NDIyMQ@@._V1_UY268_CR3,0,182,268_AL_.jpg",Two Can Play That Game,"Vivica A. Fox sizzles as a woman scorned who plans to get her man back by any means necessary. In this comedy about players and those who ""get played."" As corporate overachiever and all-around fly...","[{'id': 35, 'name': 'Comedy'}, {'id': 10749, 'name': 'Romance'}]"
38536,179029,Poor Cow (1967),NaN,NaN,NaN,NaN
16372,79842,For Neda (2010),"https://images-na.ssl-images-amazon.com/images/M/MV5BOWJjOThiNDctYTNlMC00YjQ2LThkZjEtNTRlOTk2ZmIyZjYyL2ltYWdlXkEyXkFqcGdeQXVyNTM3MDMyMDQ@._V1_UY268_CR9,0,182,268_AL_.jpg",For Neda,"On June 20, 2009, Neda Agha-Soltan was shot and killed on the streets of Tehran during the turmoil that followed the Iranian presidential contest. Within hours, images of her dying moments, captur...","[{'id': 99, 'name': 'Documentary'}]"
47661,104011,Andy Hardy's Double Life (1942),"https://images-na.ssl-images-amazon.com/images/M/MV5BMTAxMjc3ODE4MTBeQTJeQWpwZ15BbWU3MDEyNzQwMjE@._V1_UX182_CR0,0,182,268_AL_.jpg",Andy Hardy's Double Life,"Andy (Mickey Rooney) is about to head off to college but he's got a few things to take care of before leaving. For starters, he must try and sell his junk car for $20 to pay for a bill and he must...","[{'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}, {'id': 10749, 'name': 'Romance'}]"
30224,45259,Man Push Cart (2005),"https://images-na.ssl-images-amazon.com/images/M/MV5BMTYxNjcxOTkzMl5BMl5BanBnXkFtZTcwMDMwODgzMQ@@._V1_UX182_CR0,0,182,268_AL_.jpg",Man Push Cart,"Every night while the city sleeps, Ahmad, a former Pakistani rock star turned immigrant, drags his heavy cart along the streets of New York. And every morning, he sells coffee and donuts to a city...","[{'id': 18, 'name': 'Drama'}]"
16380,32799,Maidens in Uniform (Mädchen in Uniform) (1931),"https://images-na.ssl-images-amazon.com/images/M/MV5BOTA1MjYyODA2OF5BMl5BanBnXkFtZTcwMjE3MDQ5MQ@@._V1._CR0,1,353,468_UY268_CR10,0,182,268_AL_.jpg",Mädchen in Uniform,A sensitive girl is sent to an all-girls boarding school and develops a romantic attachment to one of her teachers. One of the earliest narrative films to explicitly portray homosexuality.,"[{'id': 10749, 'name': 'Romance'}, {'id': 18, 'name': 'Drama'}]"
32884,173835,Drone (2017),NaN,Drone,Ideologies collide with fatal results when a military drone contractor meets an enigmatic Pakistani businessman.,"[{'id': 53, 'name': 'Thriller'}]"
12993,32234,Julia (1977),"https://images-na.ssl-images-amazon.com/images/M/MV5BNTI2YjQ3MWYtODczNy00M2VmLWI2ODYtYmVlMzc1MTQ3MzM1XkEyXkFqcGdeQXVyMTAwMzUyOTc@._V1_UX182_CR0,0,182,268_AL_.jpg",Julia,"At the behest of an old and dear friend, playwright Lillian Hellman undertakes a dangerous mission to smuggle funds into Nazi Germany.","[{'id': 18, 'name': 'Drama'}]"
33346,187993,The Old New Year (1981),NaN,NaN,NaN,NaN


### Making data usable

In [14]:
import ast
import numpy as np
import pandas as pd

def split_pipe_genres(x):
    if pd.isna(x):
        return []
    x = str(x).strip()
    if x == "" or x == "(no genres listed)":
        return []
    return [g.strip() for g in x.split("|") if g.strip()]

def parse_tmdb_genres(x):
    if pd.isna(x):
        return []
    x = str(x).strip()
    if x == "":
        return []
    try:
        val = ast.literal_eval(x)
        if isinstance(val, list):
            out = []
            for item in val:
                if isinstance(item, dict) and "name" in item:
                    out.append(str(item["name"]).strip())
            return out
        return []
    except:
        return []

def first_non_null(*vals):
    for v in vals:
        if pd.notna(v):
            return v
    return np.nan

movie_feature_df = (
    merged_df[[
        "movieId", "title", "genres", "imdb_id_clean", "tmdb_id_clean",
        "poster_title", "poster_imdb_score", "poster_genres", "poster_url",
        "meta_title", "original_title", "meta_genres", "meta_poster_path",
        "meta_vote_average", "meta_vote_count", "meta_popularity",
        "meta_runtime", "meta_release_date", "meta_overview", "meta_original_language"
    ]]
    .drop_duplicates(subset=["movieId"])
    .reset_index(drop=True)
)

# parse genres
movie_feature_df["ml_genres_list"] = movie_feature_df["genres"].apply(split_pipe_genres)
movie_feature_df["poster_genres_list"] = movie_feature_df["poster_genres"].apply(split_pipe_genres)
movie_feature_df["meta_genres_list"] = movie_feature_df["meta_genres"].apply(parse_tmdb_genres)

# choose a cleaner final title and final genre source
movie_feature_df["final_title"] = movie_feature_df.apply(
    lambda r: first_non_null(r["meta_title"], r["poster_title"], r["title"]),
    axis=1
)

movie_feature_df["final_genres_list"] = movie_feature_df.apply(
    lambda r: r["meta_genres_list"] if len(r["meta_genres_list"]) > 0
    else (r["poster_genres_list"] if len(r["poster_genres_list"]) > 0 else r["ml_genres_list"]),
    axis=1
)

# numeric cleanup
for col in ["poster_imdb_score", "meta_vote_average", "meta_vote_count", "meta_popularity", "meta_runtime"]:
    movie_feature_df[col] = pd.to_numeric(movie_feature_df[col], errors="coerce")

movie_feature_df["meta_release_date"] = pd.to_datetime(movie_feature_df["meta_release_date"], errors="coerce")
movie_feature_df["release_year"] = movie_feature_df["meta_release_date"].dt.year

display(movie_feature_df[[
    "movieId", "title", "final_title", "ml_genres_list", "poster_genres_list",
    "meta_genres_list", "final_genres_list", "poster_url", "meta_overview"
]].head(10))

print("movie_feature_df shape:", movie_feature_df.shape)

,movieId,title,final_title,ml_genres_list,poster_genres_list,meta_genres_list,final_genres_list,poster_url,meta_overview
0,296,Pulp Fiction (1994),Pulp Fiction,"[Comedy, Crime, Drama, Thriller]","[Crime, Drama]","[Thriller, Crime]","[Thriller, Crime]","https://images-na.ssl-images-amazon.com/images/M/MV5BMTkxMTA5OTAzMl5BMl5BanBnXkFtZTgwNjA5MDc3NjE@._V1_UX182_CR0,0,182,268_AL_.jpg","A burger-loving hit man, his philosophical partner, a drug-addled gangster's moll and a washed-up boxer converge in this sprawling, comedic crime caper. Their adventures unfurl in three stories th..."
1,306,Three Colors: Red (Trois couleurs: Rouge) (1994),Three Colors: Red,[Drama],"[Drama, Mystery, Romance]","[Drama, Mystery, Romance]","[Drama, Mystery, Romance]","https://images-na.ssl-images-amazon.com/images/M/MV5BYTg1MmNiMjItMmY4Yy00ZDQ3LThjMzYtZGQ0ZTQzNTdkMGQ1L2ltYWdlL2ltYWdlXkEyXkFqcGdeQXVyMTQxNzMzNDI@._V1_UX182_CR0,0,182,268_AL_.jpg",Red This is the third film from the trilogy by Kieślowski. “Red” meaning brotherliness. Here Kieślowski masterly tells strange coincidentally linked stories in the most packed work.
2,307,Three Colors: Blue (Trois couleurs: Bleu) (1993),Three Colors: Blue,[Drama],"[Drama, Music, Mystery]","[Drama, Music, Mystery]","[Drama, Music, Mystery]","https://images-na.ssl-images-amazon.com/images/M/MV5BZjYxMzU5ZGItOTk0Yy00OGU4LTkyMGQtYmQ3MjQ0ODRjOTJlL2ltYWdlL2ltYWdlXkEyXkFqcGdeQXVyMTQxNzMzNDI@._V1_UX182_CR0,0,182,268_AL_.jpg",A woman struggles to find a way to live her life after the death of her husband and child.
3,665,Underground (1995),Underground,"[Comedy, Drama, War]","[Comedy, Drama, War]","[War, Drama, Comedy]","[War, Drama, Comedy]","https://images-na.ssl-images-amazon.com/images/M/MV5BMTgxMjI0MzY4MV5BMl5BanBnXkFtZTcwMzYxMzQyMQ@@._V1_UY268_CR3,0,182,268_AL_.jpg","Black marketeers Marko (Miki Manojlovic) and Blacky (Lazar Ristovski) manufacture and sell weapons to the Communist resistance in WWII Belgrade, living the good life along the way. Marko's surreal..."
4,899,Singin' in the Rain (1952),Singin' in the Rain,"[Comedy, Musical, Romance]","[Comedy, Musical, Romance]","[Comedy, Music, Romance]","[Comedy, Music, Romance]","https://images-na.ssl-images-amazon.com/images/M/MV5BZDRjNGViMjQtOThlMi00MTA3LThkYzQtNzJkYjBkMGE0YzE1XkEyXkFqcGdeQXVyNDYyMDk5MTU@._V1_UY268_CR0,0,182,268_AL_.jpg","In 1927 Hollywood, Don Lockwood and Lina Lamont are a famous on-screen romantic pair in silent movies, but Lina mistakes the on-screen romance for real love. When their latest film is transformed ..."
5,1088,Dirty Dancing (1987),Dirty Dancing,"[Drama, Musical, Romance]","[Drama, Music, Romance]","[Drama, Music, Romance]","[Drama, Music, Romance]","https://images-na.ssl-images-amazon.com/images/M/MV5BMTc3MDY3ODQ2OV5BMl5BanBnXkFtZTgwOTQ2NTYxMTE@._V1_UX182_CR0,0,182,268_AL_.jpg","Expecting the usual tedium that accompanies a summer in the Catskills with her family, 17-year-old Frances 'Baby' Houseman is surprised to find herself stepping into the shoes of a professional ho..."
6,1175,Delicatessen (1991),Delicatessen,"[Comedy, Drama, Romance]","[Comedy, Crime]","[Comedy, Science Fiction, Fantasy]","[Comedy, Science Fiction, Fantasy]","https://images-na.ssl-images-amazon.com/images/M/MV5BNjg5ZDM0MTEtYTZmNC00NDJiLWI5MTktYzk4N2QxY2IxZTc2L2ltYWdlXkEyXkFqcGdeQXVyNTAyODkwOQ@@._V1_UY268_CR9,0,182,268_AL_.jpg",This bizarre surrealistic black comedy takes place in a small fictitious post-apocalyptic town where food is scarce and butcher Clapet has the macabre business of using human flesh to feed his cus...
7,1217,Ran (1985),Ran,"[Drama, War]","[Action, Drama]","[Action, Drama, History]","[Action, Drama, History]","https://images-na.ssl-images-amazon.com/images/M/MV5BZDBjZTM4ZmEtOTA5ZC00NTQzLTkyNzYtMmUxNGU2YjI5YjU5L2ltYWdlXkEyXkFqcGdeQXVyNTAyODkwOQ@@._V1_UY268_CR3,0,182,268_AL_.jpg","Set in Japan in the 16th century (or so), an elderly warlord retires, handing over his empire to his three sons. However, he vastly underestimates how the new-found po

movie_feature_df shape: (59047, 26)


In [15]:
# final movie-level table
final_movie_df = movie_feature_df[[
    "movieId", "imdb_id_clean", "tmdb_id_clean",
    "title", "final_title", "original_title",
    "ml_genres_list", "poster_genres_list", "meta_genres_list", "final_genres_list",
    "poster_url", "meta_poster_path",
    "poster_imdb_score", "meta_vote_average", "meta_vote_count",
    "meta_popularity", "meta_runtime", "release_year",
    "meta_original_language", "meta_overview"
]].copy()

# simple text version of genres for easy saving/viewing
final_movie_df["final_genres_str"] = final_movie_df["final_genres_list"].apply(lambda x: "|".join(x) if isinstance(x, list) else "")

# final rating-level table for recommender
final_ratings_df = merged_df[["userId", "movieId", "rating", "timestamp"]].copy()

# join compact movie features into rating table when needed
model_df = final_ratings_df.merge(
    final_movie_df[[
        "movieId", "final_title", "final_genres_str", "poster_url",
        "poster_imdb_score", "meta_vote_average", "meta_vote_count",
        "meta_popularity", "meta_runtime", "release_year",
        "meta_original_language", "meta_overview"
    ]],
    on="movieId",
    how="left"
)

print("final_movie_df:", final_movie_df.shape)
print("final_ratings_df:", final_ratings_df.shape)
print("model_df:", model_df.shape)

display(final_movie_df.head(5))
display(model_df.head(5))

final_movie_df: (59047, 21)
final_ratings_df: (25000095, 4)
model_df: (25000095, 15)


,movieId,imdb_id_clean,tmdb_id_clean,title,final_title,original_title,ml_genres_list,poster_genres_list,meta_genres_list,final_genres_list,poster_url,meta_poster_path,poster_imdb_score,meta_vote_average,meta_vote_count,meta_popularity,meta_runtime,release_year,meta_original_language,meta_overview,final_genres_str
0,296,110912,680,Pulp Fiction (1994),Pulp Fiction,Pulp Fiction,"[Comedy, Crime, Drama, Thriller]","[Crime, Drama]","[Thriller, Crime]","[Thriller, Crime]","https://images-na.ssl-images-amazon.com/images/M/MV5BMTkxMTA5OTAzMl5BMl5BanBnXkFtZTgwNjA5MDc3NjE@._V1_UX182_CR0,0,182,268_AL_.jpg",/dM2w364MScsjFf8pfMbaWUcWrR.jpg,8.9,8.3,8670.0,140.950236,154.0,1994.0,en,"A burger-loving hit man, his philosophical partner, a drug-addled gangster's moll and a washed-up boxer converge in this sprawling, comedic crime caper. Their adventures unfurl in three stories th...",Thriller|Crime
1,306,111495,110,Three Colors: Red (Trois couleurs: Rouge) (1994),Three Colors: Red,Trois couleurs : Rouge,[Drama],"[Drama, Mystery, Romance]","[Drama, Mystery, Romance]","[Drama, Mystery, Romance]","https://images-na.ssl-images-amazon.com/images/M/MV5BYTg1MmNiMjItMmY4Yy00ZDQ3LThjMzYtZGQ0ZTQzNTdkMGQ1L2ltYWdlL2ltYWdlXkEyXkFqcGdeQXVyMTQxNzMzNDI@._V1_UX182_CR0,0,182,268_AL_.jpg",/77CFEssoKesi4zvtADEpIrSKhA3.jpg,8.1,7.8,246.0,7.832755,99.0,1994.0,fr,Red This is the third film from the trilogy by Kieślowski. “Red” meaning brotherliness. Here Kieślowski masterly tells strange coincidentally linked stories in the most packed work.,Drama|Mystery|Romance
2,307,108394,108,Three Colors: Blue (Trois couleurs: Bleu) (1993),Three Colors: Blue,Trois couleurs : Bleu,[Drama],"[Drama, Music, Mystery]","[Drama, Music, Mystery]","[Drama, Music, Mystery]","https://images-na.ssl-images-amazon.com/images/M/MV5BZjYxMzU5ZGItOTk0Yy00OGU4LTkyMGQtYmQ3MjQ0ODRjOTJlL2ltYWdlL2ltYWdlXkEyXkFqcGdeQXVyMTQxNzMzNDI@._V1_UX182_CR0,0,182,268_AL_.jpg",/ztIqf7yGmb4JRFiv2i62K6PhTQR.jpg,8.0,7.7,311.0,8.843517,98.0,1993.0,fr,A woman struggles to find a way to live her life after the death of her husband and child.,Drama|Music|Mystery
3,665,114787,11902,Underground (1995),Underground,Podzemlje,"[Comedy, Drama, War]","[Comedy, Drama, War]","[War, Drama, Comedy]","[War, Drama, Comedy]","https://images-na.ssl-images-amazon.com/images/M/MV5BMTgxMjI0MzY4MV5BMl5BanBnXkFtZTcwMzYxMzQyMQ@@._V1_UY268_CR3,0,182,268_AL_.jpg",/bnrLsunhGI5FO316sSThtAHikdV.jpg,8.1,7.5,143.0,14.587894,170.0,1995.0,sr,"Black marketeers Marko (Miki Manojlovic) and Blacky (Lazar Ristovski) manufacture and sell weapons to the Communist resistance in WWII Belgrade, living the good life along the way. Marko's surreal...",War|Drama|Comedy
4,899,45152,872,Singin' in the Rain (1952),Singin' in the Rain,Singin' in the Rain,"[Comedy, Musical, Romance]","[Comedy, Musical, Romance]","[Comedy, Music, Romance]","[Comedy, Music, Romance]","https://images-na.ssl-images-amazon.com/images/M/MV5BZDRjNGViMjQtOThlMi00MTA3LThkYzQtNzJkYjBkMGE0YzE1XkEyXkFqcGdeQXVyNDYyMDk5MTU@._V1_UY268_CR0,0,182,268_AL_.jpg",/d5J53CwrVs6txB8zhE6qS2QhIV.jpg,8.3,7.9,747.0,11.064858,103.0,1952.0,en,"In 1927 Hollywood, Don Lockwood and Lina Lamont are a famous on-screen romantic pair in silent movies, but Lina mistakes the on-screen romance for real love. When their latest film is transformed ...",Comedy|Music|Romance


,userId,movieId,rating,timestamp,final_title,final_genres_str,poster_url,poster_imdb_score,meta_vote_average,meta_vote_count,meta_popularity,meta_runtime,release_year,meta_original_language,meta_overview
0,1,296,5.0,1147880044,Pulp Fiction,Thriller|Crime,"https://images-na.ssl-images-amazon.com/images/M/MV5BMTkxMTA5OTAzMl5BMl5BanBnXkFtZTgwNjA5MDc3NjE@._V1_UX182_CR0,0,182,268_AL_.jpg",8.9,8.3,8670.0,140.950236,154.0,1994.0,en,"A burger-loving hit man, his philosophical partner, a drug-addled gangster's moll and a washed-up boxer converge in this sprawling, comedic crime caper. Their adventures unfurl in three stories th..."
1,1,306,3.5,1147868817,Three Colors: Red,Drama|Mystery|Romance,"https://images-na.ssl-images-amazon.com/images/M/MV5BYTg1MmNiMjItMmY4Yy00ZDQ3LThjMzYtZGQ0ZTQzNTdkMGQ1L2ltYWdlL2ltYWdlXkEyXkFqcGdeQXVyMTQxNzMzNDI@._V1_UX182_CR0,0,182,268_AL_.jpg",8.1,7.8,246.0,7.832755,99.0,1994.0,fr,Red This is the third film from the trilogy by Kieślowski. “Red” meaning brotherliness. Here Kieślowski masterly tells strange coincidentally linked stories in the most packed work.
2,1,307,5.0,1147868828,Three Colors: Blue,Drama|Music|Mystery,"https://images-na.ssl-images-amazon.com/images/M/MV5BZjYxMzU5ZGItOTk0Yy00OGU4LTkyMGQtYmQ3MjQ0ODRjOTJlL2ltYWdlL2ltYWdlXkEyXkFqcGdeQXVyMTQxNzMzNDI@._V1_UX182_CR0,0,182,268_AL_.jpg",8.0,7.7,311.0,8.843517,98.0,1993.0,fr,A woman struggles to find a way to live her life after the death of her husband and child.
3,1,665,5.0,1147878820,Underground,War|Drama|Comedy,"https://images-na.ssl-images-amazon.com/images/M/MV5BMTgxMjI0MzY4MV5BMl5BanBnXkFtZTcwMzYxMzQyMQ@@._V1_UY268_CR3,0,182,268_AL_.jpg",8.1,7.5,143.0,14.587894,170.0,1995.0,sr,"Black marketeers Marko (Miki Manojlovic) and Blacky (Lazar Ristovski) manufacture and sell weapons to the Communist resistance in WWII Belgrade, living the good life along the way. Marko's surreal..."
4,1,899,3.5,1147868510,Singin' in the Rain,Comedy|Music|Romance,"https://images-na.ssl-images-amazon.com/images/M/MV5BZDRjNGViMjQtOThlMi00MTA3LThkYzQtNzJkYjBkMGE0YzE1XkEyXkFqcGdeQXVyNDYyMDk5MTU@._V1_UY268_CR0,0,182,268_AL_.jpg",8.3,7.9,747.0,11.064858,103.0,1952.0,en,"In 1927 Hollywood, Don Lockwood and Lina Lamont are a famous on-screen romantic pair in silent movies, but Lina mistakes the on-screen romance for real love. When their latest film is transformed ..."


In [16]:
# quick sanity stats
stats = pd.DataFrame({
    "metric": [
        "unique users",
        "unique movies in ratings",
        "movies with poster_url",
        "movies with overview",
        "movies with final genres",
        "movies with poster + overview"
    ],
    "value": [
        final_ratings_df["userId"].nunique(),
        final_ratings_df["movieId"].nunique(),
        final_movie_df["poster_url"].notna().sum(),
        final_movie_df["meta_overview"].notna().sum(),
        final_movie_df["final_genres_list"].apply(len).gt(0).sum(),
        ((final_movie_df["poster_url"].notna()) & (final_movie_df["meta_overview"].notna())).sum()
    ]
})

display(stats)

# save
final_movie_df.to_csv("final_movie_features.csv", index=False)
final_ratings_df.to_csv("final_ratings_only.csv", index=False)

# smaller sample for fast experiments
model_sample_df = model_df.sample(min(500000, len(model_df)), random_state=42)
model_sample_df.to_csv("model_sample_500k.csv", index=False)

print("Saved:")
print("- final_movie_features.csv")
print("- final_ratings_only.csv")
print("- model_sample_500k.csv")

,metric,value
0,unique users,162541
1,unique movies in ratings,59047
2,movies with poster_url,35912
3,movies with overview,41477
4,movies with final genres,56628
5,movies with poster + overview,35146


Saved:
- final_movie_features.csv
- final_ratings_only.csv
- model_sample_500k.csv


### Final preprocessing, normalization

In [19]:
import ast
import numpy as np
import pandas as pd

# ---------- helpers ----------
genre_map = {
    "Musical": "Music",
    "Sci-Fi": "Science Fiction",
    "Children": "Family",
    "Film-Noir": "Noir"
}

def standardize_genres(genre_list):
    if not isinstance(genre_list, list):
        return []
    out = []
    for g in genre_list:
        g = str(g).strip()
        if g == "":
            continue
        g2 = genre_map.get(g, g)
        if g2 not in out:
            out.append(g2)
    return out

def combine_genres(row):
    combined = []
    for source in ["meta_genres_list", "poster_genres_list", "ml_genres_list"]:
        vals = row[source]
        if not isinstance(vals, list):
            continue
        for g in vals:
            if g not in combined:
                combined.append(g)
    return combined

def clean_text(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    return x if x != "" else np.nan

# ---------- normalize text ----------
for col in ["title", "final_title", "original_title", "meta_overview", "meta_original_language", "poster_url", "meta_poster_path"]:
    if col in final_movie_df.columns:
        final_movie_df[col] = final_movie_df[col].apply(clean_text)

# ---------- normalize genre lists ----------
for col in ["ml_genres_list", "poster_genres_list", "meta_genres_list", "final_genres_list"]:
    if col in final_movie_df.columns:
        final_movie_df[col] = final_movie_df[col].apply(standardize_genres)

# ---------- combine all genre sources ----------
final_movie_df["final_genres_list"] = final_movie_df.apply(combine_genres, axis=1)
final_movie_df["final_genres_str"] = final_movie_df["final_genres_list"].apply(lambda x: "|".join(x) if isinstance(x, list) else "")

# ---------- numeric cleanup ----------
numeric_cols = [
    "poster_imdb_score", "meta_vote_average", "meta_vote_count",
    "meta_popularity", "meta_runtime", "release_year"
]

for col in numeric_cols:
    if col in final_movie_df.columns:
        final_movie_df[col] = pd.to_numeric(final_movie_df[col], errors="coerce")

# ---------- fill some missing numeric values ----------
for col in ["poster_imdb_score", "meta_vote_average", "meta_vote_count", "meta_popularity", "meta_runtime"]:
    if col in final_movie_df.columns:
        final_movie_df[col] = final_movie_df[col].fillna(final_movie_df[col].median())

print("Normalization done.")
display(final_movie_df[[
    "movieId", "final_title", "ml_genres_list", "poster_genres_list",
    "meta_genres_list", "final_genres_list", "final_genres_str"
]].head(10))

Normalization done.


,movieId,final_title,ml_genres_list,poster_genres_list,meta_genres_list,final_genres_list,final_genres_str
0,296,Pulp Fiction,"[Comedy, Crime, Drama, Thriller]","[Crime, Drama]","[Thriller, Crime]","[Thriller, Crime, Drama, Comedy]",Thriller|Crime|Drama|Comedy
1,306,Three Colors: Red,[Drama],"[Drama, Mystery, Romance]","[Drama, Mystery, Romance]","[Drama, Mystery, Romance]",Drama|Mystery|Romance
2,307,Three Colors: Blue,[Drama],"[Drama, Music, Mystery]","[Drama, Music, Mystery]","[Drama, Music, Mystery]",Drama|Music|Mystery
3,665,Underground,"[Comedy, Drama, War]","[Comedy, Drama, War]","[War, Drama, Comedy]","[War, Drama, Comedy]",War|Drama|Comedy
4,899,Singin' in the Rain,"[Comedy, Music, Romance]","[Comedy, Music, Romance]","[Comedy, Music, Romance]","[Comedy, Music, Romance]",Comedy|Music|Romance
5,1088,Dirty Dancing,"[Drama, Music, Romance]","[Drama, Music, Romance]","[Drama, Music, Romance]","[Drama, Music, Romance]",Drama|Music|Romance
6,1175,Delicatessen,"[Comedy, Drama, Romance]","[Comedy, Crime]","[Comedy, Science Fiction, Fantasy]","[Comedy, Science Fiction, Fantasy, Crime, Drama, Romance]",Comedy|Science Fiction|Fantasy|Crime|Drama|Romance
7,1217,Ran,"[Drama, War]","[Action, Drama]","[Action, Drama, History]","[Action, Drama, History, War]",Action|Drama|History|War
8,1237,The Seventh Seal,[Drama],"[Drama, Fantasy]","[Fantasy, Drama]","[Fantasy, Drama]",Fantasy|Drama
9,1250,The Bridge on the River Kwai,"[Adventure, Drama, War]","[Adventure, Drama, War]","[Drama, History, War]","[Drama, History, War, Adventure]",Drama|History|War|Adventure


In [20]:
# ---------- create mappings ----------
user_ids = sorted(final_ratings_df["userId"].unique())
movie_ids = sorted(final_movie_df["movieId"].unique())

user2idx = pd.DataFrame({
    "userId": user_ids,
    "user_idx": range(len(user_ids))
})

movie2idx = pd.DataFrame({
    "movieId": movie_ids,
    "movie_idx": range(len(movie_ids))
})

# ---------- attach indices ----------
final_ratings_df = final_ratings_df.merge(user2idx, on="userId", how="left")
final_ratings_df = final_ratings_df.merge(movie2idx, on="movieId", how="left")
final_movie_df = final_movie_df.merge(movie2idx, on="movieId", how="left")

print("num users:", len(user2idx))
print("num movies:", len(movie2idx))

display(user2idx.head())
display(movie2idx.head())
display(final_ratings_df.head())
display(final_movie_df.head())

num users: 162541
num movies: 59047


,userId,user_idx
0,1,0
1,2,1
2,3,2
3,4,3
4,5,4


,movieId,movie_idx
0,1,0
1,2,1
2,3,2
3,4,3
4,5,4


,userId,movieId,rating,timestamp,user_idx,movie_idx
0,1,296,5.0,1147880044,0,292
1,1,306,3.5,1147868817,0,302
2,1,307,5.0,1147868828,0,303
3,1,665,5.0,1147878820,0,654
4,1,899,3.5,1147868510,0,878


,movieId,imdb_id_clean,tmdb_id_clean,title,final_title,original_title,ml_genres_list,poster_genres_list,meta_genres_list,final_genres_list,poster_url,meta_poster_path,poster_imdb_score,meta_vote_average,meta_vote_count,meta_popularity,meta_runtime,release_year,meta_original_language,meta_overview,final_genres_str,movie_idx
0,296,110912,680,Pulp Fiction (1994),Pulp Fiction,Pulp Fiction,"[Comedy, Crime, Drama, Thriller]","[Crime, Drama]","[Thriller, Crime]","[Thriller, Crime, Drama, Comedy]","https://images-na.ssl-images-amazon.com/images/M/MV5BMTkxMTA5OTAzMl5BMl5BanBnXkFtZTgwNjA5MDc3NjE@._V1_UX182_CR0,0,182,268_AL_.jpg",/dM2w364MScsjFf8pfMbaWUcWrR.jpg,8.9,8.3,8670.0,140.950236,154.0,1994.0,en,"A burger-loving hit man, his philosophical partner, a drug-addled gangster's moll and a washed-up boxer converge in this sprawling, comedic crime caper. Their adventures unfurl in three stories th...",Thriller|Crime|Drama|Comedy,292
1,306,111495,110,Three Colors: Red (Trois couleurs: Rouge) (1994),Three Colors: Red,Trois couleurs : Rouge,[Drama],"[Drama, Mystery, Romance]","[Drama, Mystery, Romance]","[Drama, Mystery, Romance]","https://images-na.ssl-images-amazon.com/images/M/MV5BYTg1MmNiMjItMmY4Yy00ZDQ3LThjMzYtZGQ0ZTQzNTdkMGQ1L2ltYWdlL2ltYWdlXkEyXkFqcGdeQXVyMTQxNzMzNDI@._V1_UX182_CR0,0,182,268_AL_.jpg",/77CFEssoKesi4zvtADEpIrSKhA3.jpg,8.1,7.8,246.0,7.832755,99.0,1994.0,fr,Red This is the third film from the trilogy by Kieślowski. “Red” meaning brotherliness. Here Kieślowski masterly tells strange coincidentally linked stories in the most packed work.,Drama|Mystery|Romance,302
2,307,108394,108,Three Colors: Blue (Trois couleurs: Bleu) (1993),Three Colors: Blue,Trois couleurs : Bleu,[Drama],"[Drama, Music, Mystery]","[Drama, Music, Mystery]","[Drama, Music, Mystery]","https://images-na.ssl-images-amazon.com/images/M/MV5BZjYxMzU5ZGItOTk0Yy00OGU4LTkyMGQtYmQ3MjQ0ODRjOTJlL2ltYWdlL2ltYWdlXkEyXkFqcGdeQXVyMTQxNzMzNDI@._V1_UX182_CR0,0,182,268_AL_.jpg",/ztIqf7yGmb4JRFiv2i62K6PhTQR.jpg,8.0,7.7,311.0,8.843517,98.0,1993.0,fr,A woman struggles to find a way to live her life after the death of her husband and child.,Drama|Music|Mystery,303
3,665,114787,11902,Underground (1995),Underground,Podzemlje,"[Comedy, Drama, War]","[Comedy, Drama, War]","[War, Drama, Comedy]","[War, Drama, Comedy]","https://images-na.ssl-images-amazon.com/images/M/MV5BMTgxMjI0MzY4MV5BMl5BanBnXkFtZTcwMzYxMzQyMQ@@._V1_UY268_CR3,0,182,268_AL_.jpg",/bnrLsunhGI5FO316sSThtAHikdV.jpg,8.1,7.5,143.0,14.587894,170.0,1995.0,sr,"Black marketeers Marko (Miki Manojlovic) and Blacky (Lazar Ristovski) manufacture and sell weapons to the Communist resistance in WWII Belgrade, living the good life along the way. Marko's surreal...",War|Drama|Comedy,654
4,899,45152,872,Singin' in the Rain (1952),Singin' in the Rain,Singin' in the Rain,"[Comedy, Music, Romance]","[Comedy, Music, Romance]","[Comedy, Music, Romance]","[Comedy, Music, Romance]","https://images-na.ssl-images-amazon.com/images/M/MV5BZDRjNGViMjQtOThlMi00MTA3LThkYzQtNzJkYjBkMGE0YzE1XkEyXkFqcGdeQXVyNDYyMDk5MTU@._V1_UY268_CR0,0,182,268_AL_.jpg",/d5J53CwrVs6txB8zhE6qS2QhIV.jpg,8.3,7.9,747.0,11.064858,103.0,1952.0,en,"In 1927 Hollywood, Don Lockwood and Lina Lamont are a famous on-screen romantic pair in silent movies, but Lina mistakes the on-screen romance for real love. When their latest film is transformed ...",Comedy|Music|Romance,878


In [21]:
model_df = final_ratings_df.merge(
    final_movie_df[[
        "movieId", "movie_idx", "final_title", "final_genres_str",
        "poster_url", "poster_imdb_score", "meta_vote_average",
        "meta_vote_count", "meta_popularity", "meta_runtime",
        "release_year", "meta_original_language", "meta_overview"
    ]],
    on=["movieId", "movie_idx"],
    how="left"
)

print("final_movie_df:", final_movie_df.shape)
print("final_ratings_df:", final_ratings_df.shape)
print("model_df:", model_df.shape)

display(model_df.head())

final_movie_df: (59047, 22)
final_ratings_df: (25000095, 6)
model_df: (25000095, 17)


,userId,movieId,rating,timestamp,user_idx,movie_idx,final_title,final_genres_str,poster_url,poster_imdb_score,meta_vote_average,meta_vote_count,meta_popularity,meta_runtime,release_year,meta_original_language,meta_overview
0,1,296,5.0,1147880044,0,292,Pulp Fiction,Thriller|Crime|Drama|Comedy,"https://images-na.ssl-images-amazon.com/images/M/MV5BMTkxMTA5OTAzMl5BMl5BanBnXkFtZTgwNjA5MDc3NjE@._V1_UX182_CR0,0,182,268_AL_.jpg",8.9,8.3,8670.0,140.950236,154.0,1994.0,en,"A burger-loving hit man, his philosophical partner, a drug-addled gangster's moll and a washed-up boxer converge in this sprawling, comedic crime caper. Their adventures unfurl in three stories th..."
1,1,306,3.5,1147868817,0,302,Three Colors: Red,Drama|Mystery|Romance,"https://images-na.ssl-images-amazon.com/images/M/MV5BYTg1MmNiMjItMmY4Yy00ZDQ3LThjMzYtZGQ0ZTQzNTdkMGQ1L2ltYWdlL2ltYWdlXkEyXkFqcGdeQXVyMTQxNzMzNDI@._V1_UX182_CR0,0,182,268_AL_.jpg",8.1,7.8,246.0,7.832755,99.0,1994.0,fr,Red This is the third film from the trilogy by Kieślowski. “Red” meaning brotherliness. Here Kieślowski masterly tells strange coincidentally linked stories in the most packed work.
2,1,307,5.0,1147868828,0,303,Three Colors: Blue,Drama|Music|Mystery,"https://images-na.ssl-images-amazon.com/images/M/MV5BZjYxMzU5ZGItOTk0Yy00OGU4LTkyMGQtYmQ3MjQ0ODRjOTJlL2ltYWdlL2ltYWdlXkEyXkFqcGdeQXVyMTQxNzMzNDI@._V1_UX182_CR0,0,182,268_AL_.jpg",8.0,7.7,311.0,8.843517,98.0,1993.0,fr,A woman struggles to find a way to live her life after the death of her husband and child.
3,1,665,5.0,1147878820,0,654,Underground,War|Drama|Comedy,"https://images-na.ssl-images-amazon.com/images/M/MV5BMTgxMjI0MzY4MV5BMl5BanBnXkFtZTcwMzYxMzQyMQ@@._V1_UY268_CR3,0,182,268_AL_.jpg",8.1,7.5,143.0,14.587894,170.0,1995.0,sr,"Black marketeers Marko (Miki Manojlovic) and Blacky (Lazar Ristovski) manufacture and sell weapons to the Communist resistance in WWII Belgrade, living the good life along the way. Marko's surreal..."
4,1,899,3.5,1147868510,0,878,Singin' in the Rain,Comedy|Music|Romance,"https://images-na.ssl-images-amazon.com/images/M/MV5BZDRjNGViMjQtOThlMi00MTA3LThkYzQtNzJkYjBkMGE0YzE1XkEyXkFqcGdeQXVyNDYyMDk5MTU@._V1_UY268_CR0,0,182,268_AL_.jpg",8.3,7.9,747.0,11.064858,103.0,1952.0,en,"In 1927 Hollywood, Don Lockwood and Lina Lamont are a famous on-screen romantic pair in silent movies, but Lina mistakes the on-screen romance for real love. When their latest film is transformed ..."


In [22]:
all_genres = sorted(set(g for lst in final_movie_df["final_genres_list"] for g in lst))
genre2idx = {g: i for i, g in enumerate(all_genres)}

def to_multihot(genre_list, genre2idx):
    vec = np.zeros(len(genre2idx), dtype=int)
    if isinstance(genre_list, list):
        for g in genre_list:
            if g in genre2idx:
                vec[genre2idx[g]] = 1
    return vec

final_movie_df["genre_multihot"] = final_movie_df["final_genres_list"].apply(lambda x: to_multihot(x, genre2idx))

print("Number of unique genres:", len(all_genres))
print(all_genres)

display(final_movie_df[["movieId", "final_title", "final_genres_list", "genre_multihot"]].head())

Number of unique genres: 30
['Action', 'Adult', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'Foreign', 'Game-Show', 'History', 'Horror', 'IMAX', 'Music', 'Mystery', 'News', 'Noir', 'Reality-TV', 'Romance', 'Science Fiction', 'Short', 'Sport', 'TV Movie', 'Talk-Show', 'Thriller', 'War', 'Western']


,movieId,final_title,final_genres_list,genre_multihot
0,296,Pulp Fiction,"[Thriller, Crime, Drama, Comedy]","[0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]"
1,306,Three Colors: Red,"[Drama, Mystery, Romance]","[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]"
2,307,Three Colors: Blue,"[Drama, Music, Mystery]","[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
3,665,Underground,"[War, Drama, Comedy]","[0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]"
4,899,Singin' in the Rain,"[Comedy, Music, Romance]","[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]"


In [ ]:
# main files
final_movie_df.to_csv("final_movie_features_clean.csv", index=False)
final_ratings_df.to_csv("final_ratings_clean.csv", index=False)
model_df.to_csv("model_df_clean.csv", index=False)

# mappings
user2idx.to_csv("user2idx.csv", index=False)
movie2idx.to_csv("movie2idx.csv", index=False)

# optional smaller sample for fast experiments
model_sample_df = model_df.sample(min(500000, len(model_df)), random_state=42)
model_sample_df.to_csv("model_sample_500k.csv", index=False)

# optional parquet versions
#final_movie_df.to_parquet("final_movie_features_clean.parquet", index=False)
#final_ratings_df.to_parquet("final_ratings_clean.parquet", index=False)
#model_df.to_parquet("model_df_clean.parquet", index=False)

print("Saved:")
print("- final_movie_features_clean.csv")
print("- final_ratings_clean.csv")
print("- model_df_clean.csv")
print("- model_sample_500k.csv")
print("- user2idx.csv")
print("- movie2idx.csv")
print("- final_movie_features_clean.parquet")
print("- final_ratings_clean.parquet")
print("- model_df_clean.parquet")

In [24]:
stats = pd.DataFrame({
    "metric": [
        "unique users",
        "unique movies",
        "ratings rows",
        "movies with poster",
        "movies with overview",
        "movies with genres",
        "movies with poster + overview"
    ],
    "value": [
        final_ratings_df["userId"].nunique(),
        final_movie_df["movieId"].nunique(),
        len(final_ratings_df),
        final_movie_df["poster_url"].notna().sum(),
        final_movie_df["meta_overview"].notna().sum(),
        final_movie_df["final_genres_list"].apply(len).gt(0).sum(),
        ((final_movie_df["poster_url"].notna()) & (final_movie_df["meta_overview"].notna())).sum()
    ]
})

display(stats)
display(final_movie_df[[
    "movieId", "movie_idx", "final_title", "final_genres_str",
    "poster_url", "meta_vote_average", "meta_runtime", "release_year"
]].head(10))

,metric,value
0,unique users,162541
1,unique movies,59047
2,ratings rows,25000095
3,movies with poster,35912
4,movies with overview,41472
5,movies with genres,56628
6,movies with poster + overview,35142


,movieId,movie_idx,final_title,final_genres_str,poster_url,meta_vote_average,meta_runtime,release_year
0,296,292,Pulp Fiction,Thriller|Crime|Drama|Comedy,"https://images-na.ssl-images-amazon.com/images/M/MV5BMTkxMTA5OTAzMl5BMl5BanBnXkFtZTgwNjA5MDc3NjE@._V1_UX182_CR0,0,182,268_AL_.jpg",8.3,154.0,1994.0
1,306,302,Three Colors: Red,Drama|Mystery|Romance,"https://images-na.ssl-images-amazon.com/images/M/MV5BYTg1MmNiMjItMmY4Yy00ZDQ3LThjMzYtZGQ0ZTQzNTdkMGQ1L2ltYWdlL2ltYWdlXkEyXkFqcGdeQXVyMTQxNzMzNDI@._V1_UX182_CR0,0,182,268_AL_.jpg",7.8,99.0,1994.0
2,307,303,Three Colors: Blue,Drama|Music|Mystery,"https://images-na.ssl-images-amazon.com/images/M/MV5BZjYxMzU5ZGItOTk0Yy00OGU4LTkyMGQtYmQ3MjQ0ODRjOTJlL2ltYWdlL2ltYWdlXkEyXkFqcGdeQXVyMTQxNzMzNDI@._V1_UX182_CR0,0,182,268_AL_.jpg",7.7,98.0,1993.0
3,665,654,Underground,War|Drama|Comedy,"https://images-na.ssl-images-amazon.com/images/M/MV5BMTgxMjI0MzY4MV5BMl5BanBnXkFtZTcwMzYxMzQyMQ@@._V1_UY268_CR3,0,182,268_AL_.jpg",7.5,170.0,1995.0
4,899,878,Singin' in the Rain,Comedy|Music|Romance,"https://images-na.ssl-images-amazon.com/images/M/MV5BZDRjNGViMjQtOThlMi00MTA3LThkYzQtNzJkYjBkMGE0YzE1XkEyXkFqcGdeQXVyNDYyMDk5MTU@._V1_UY268_CR0,0,182,268_AL_.jpg",7.9,103.0,1952.0
5,1088,1061,Dirty Dancing,Drama|Music|Romance,"https://images-na.ssl-images-amazon.com/images/M/MV5BMTc3MDY3ODQ2OV5BMl5BanBnXkFtZTgwOTQ2NTYxMTE@._V1_UX182_CR0,0,182,268_AL_.jpg",7.1,100.0,1987.0
6,1175,1147,Delicatessen,Comedy|Science Fiction|Fantasy|Crime|Drama|Romance,"https://images-na.ssl-images-amazon.com/images/M/MV5BNjg5ZDM0MTEtYTZmNC00NDJiLWI5MTktYzk4N2QxY2IxZTc2L2ltYWdlXkEyXkFqcGdeQXVyNTAyODkwOQ@@._V1_UY268_CR9,0,182,268_AL_.jpg",7.4,99.0,1991.0
7,1217,1186,Ran,Action|Drama|History|War,"https://images-na.ssl-images-amazon.com/images/M/MV5BZDBjZTM4ZmEtOTA5ZC00NTQzLTkyNzYtMmUxNGU2YjI5YjU5L2ltYWdlXkEyXkFqcGdeQXVyNTAyODkwOQ@@._V1_UY268_CR3,0,182,268_AL_.jpg",7.9,162.0,1985.0
8,1237,1205,The Seventh Seal,Fantasy|Drama,"https://images-na.ssl-images-amazon.com/images/M/MV5BM2I1ZWU4YjMtYzU0My00YmMzLWFmNTAtZDJhZGYwMmI3YWQ5XkEyXkFqcGdeQXVyNjU0OTQ0OTY@._V1_UX182_CR0,0,182,268_AL_.jpg",7.9,96.0,1957.0
9,1250,1217,The Bridge on the River Kwai,Drama|History|War|Adventure,"https://images-na.ssl-images-amazon.com/images/M/MV5BYjE1MzU4NWEtYjhjZS00NzM3LWIyYjQtMWU1M2VkNWYzMzMwXkEyXkFqcGdeQXVyNjc1NTYyMjg@._V1_UX182_CR0,0,182,268_AL_.jpg",7.7,161.0,1957.0


In [ ]:
import os
import pandas as pd

base_dir = "./MultimodalMovieDataset"

final_files = [
    "final_movie_features_clean.csv",
    "final_ratings_clean.csv",
    "model_df_clean.csv",
    "model_sample_500k.csv",
    "movie2idx.csv",
    "user2idx.csv"
]

for file_name in final_files:
    path = os.path.join(base_dir, file_name)
    print("=" * 130)
    print(f"FILE: {file_name}")
    
    if not os.path.exists(path):
        print("Not found:", path)
        continue
    
    df = pd.read_csv(path, low_memory=False)
    
    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())
    
    print("\nDtypes:")
    display(df.dtypes.to_frame("dtype").reset_index().rename(columns={"index": "column"}))
    
    print("\nMissing values:")
    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    display(missing.to_frame("missing_count") if len(missing) > 0 else pd.DataFrame({"missing_count": []}))
    
    print("\nFirst 3 rows:")
    display(df.head(3))

FILE: final_movie_features_clean.csv
Shape: (59047, 23)

Columns:
['movieId', 'imdb_id_clean', 'tmdb_id_clean', 'title', 'final_title', 'original_title', 'ml_genres_list', 'poster_genres_list', 'meta_genres_list', 'final_genres_list', 'poster_url', 'meta_poster_path', 'poster_imdb_score', 'meta_vote_average', 'meta_vote_count', 'meta_popularity', 'meta_runtime', 'release_year', 'meta_original_language', 'meta_overview', 'final_genres_str', 'movie_idx', 'genre_multihot']

Dtypes:


,column,dtype
0,movieId,int64
1,imdb_id_clean,int64
2,tmdb_id_clean,float64
3,title,object
4,final_title,object
5,original_title,object
6,ml_genres_list,object
7,poster_genres_list,object
8,meta_genres_list,object
9,final_genres_list,object



Missing values:


,missing_count
poster_url,23135
meta_overview,17575
meta_poster_path,17064
release_year,16805
meta_original_language,16744
original_title,16734
final_genres_str,2419
tmdb_id_clean,102



First 3 rows:


...


Missing values:


,missing_count



First 3 rows:


,userId,movieId,rating,timestamp,user_idx,movie_idx
0,1,296,5.0,1147880044,0,292
1,1,306,3.5,1147868817,0,302
2,1,307,5.0,1147868828,0,303


FILE: model_df_clean.csv


In [ ]:
import os
import pandas as pd

base_dir = "./MultimodalMovieDataset"

final_files = [
    "final_movie_features_clean.csv",
    "final_ratings_clean.csv",
    "model_df_clean.csv",
    "model_sample_500k.csv",
    "movie2idx.csv",
    "user2idx.csv"
]

for file_name in final_files:
    path = os.path.join(base_dir, file_name)
    print("=" * 130)
    print(f"FILE: {file_name}")
    
    if not os.path.exists(path):
        print("Not found:", path)
        continue
    
    try:
        # Read only the first few rows to get header and an idea of the data
        df_head = pd.read_csv(path, nrows=5, low_memory=False)
        
        print("Shape: (approximate, getting full row count is memory intensive)")
        # Fast way to count rows
        with open(path, 'r', encoding='utf-8') as f:
            num_rows = sum(1 for l in f) - 1 # subtract header
        print(f"({num_rows}, {len(df_head.columns)})")

        print("\nColumns:")
        print(df_head.columns.tolist())
        
        print("\nDtypes (inferred from first 1000 rows):")
        df_sample = pd.read_csv(path, nrows=1000, low_memory=False)
        display(df_sample.dtypes.to_frame("dtype").reset_index().rename(columns={"index": "column"}))
        
        print("\nMissing values (from sample of 100,000 rows):")
        chunk_iter = pd.read_csv(path, chunksize=100000, low_memory=False)
        first_chunk = next(chunk_iter)
        missing = first_chunk.isna().sum()
        missing = missing[missing > 0].sort_values(ascending=False)
        display(missing.to_frame("missing_count") if len(missing) > 0 else pd.DataFrame({"missing_count": []}))

        print("\nFirst 3 rows:")
        display(df_head.head(3))

    except Exception as e:
        print(f"An error occurred: {e}")


FILE: final_movie_features_clean.csv
Shape: (approximate, getting full row count is memory intensive)
(59136, 23)

Columns:
['movieId', 'imdb_id_clean', 'tmdb_id_clean', 'title', 'final_title', 'original_title', 'ml_genres_list', 'poster_genres_list', 'meta_genres_list', 'final_genres_list', 'poster_url', 'meta_poster_path', 'poster_imdb_score', 'meta_vote_average', 'meta_vote_count', 'meta_popularity', 'meta_runtime', 'release_year', 'meta_original_language', 'meta_overview', 'final_genres_str', 'movie_idx', 'genre_multihot']

Dtypes (inferred from first 1000 rows):


,column,dtype
0,movieId,int64
1,imdb_id_clean,int64
2,tmdb_id_clean,int64
3,title,object
4,final_title,object
5,original_title,object
6,ml_genres_list,object
7,poster_genres_list,object
8,meta_genres_list,object
9,final_genres_list,object



Missing values (from sample of 100,000 rows):


,missing_count
poster_url,23135
meta_overview,17575
meta_poster_path,17064
release_year,16805
meta_original_language,16744
original_title,16734
final_genres_str,2419
tmdb_id_clean,102



First 3 rows:


...


Missing values (from sample of 100,000 rows):


,missing_count



First 3 rows:


,userId,user_idx
0,1,0
1,2,1
2,3,2


: 

### MultimodalMovieDataset_v2 repair

The original merge loaded `TMDB_all_movies.csv` but did not use it as a fallback.
Run the cell below to build `MultimodalMovieDataset_v2`, which fills poster and metadata gaps from TMDB-all while preserving the existing MovieLens index mappings.


In [ ]:
from pathlib import Path
import subprocess
import sys

script = Path('build_multimodal_dataset_v2.py')
if not script.exists():
    script = Path('dataset') / 'build_multimodal_dataset_v2.py'
subprocess.check_call([sys.executable, str(script)])
